In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression

In [ ]:
df = pd.read_csv("../Data/Titanic.csv")

df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

### Initial Observations

From the dataset inspection we observe:

1. The dataset contains 891 passenger records.
2. Some columns contain missing values such as Age, Cabin, and Embarked.
3. Cabin has a large number of missing values.
4. The dataset contains both numerical and categorical features.
5. Data cleaning will be required before training machine learning models.

### Feature Selection

Some columns in the dataset do not provide useful information for predicting survival.

These columns are removed before training the machine learning models.

Columns dropped:

- PassengerId → only an identifier for passengers
- Name → mostly unique values and not directly useful
- Ticket → ticket numbers do not provide meaningful predictive information
- Cabin → contains too many missing values

In [ ]:
df = df.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1)

df.head()

In [ ]:
print(f"Shape : {df.shape}")
print(f"Col : {df.columns}")

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

### Handling Missing Values

Machine learning models cannot handle missing values, so we must fill them before training the models.

In the Titanic dataset:

- Age has 177 missing values
- Embarked has 2 missing values

We will handle them as follows:

- Age → fill with the median age
- Embarked → fill with the most frequent value (mode)

In [ ]:
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

In [ ]:
df.isna().sum()

In [ ]:
print(df["Sex"].value_counts())
df["Embarked"].value_counts()

In [ ]:
df.sample(5)

### Categorical Encoding

Machine learning models require numerical input, but some features in the dataset contain categorical values.

In the Titanic dataset, the following columns are categorical:

- Sex
- Embarked

We convert these categorical values into numerical form so that machine learning algorithms can process them.

In [ ]:
df["Sex"] = df["Sex"].map({'male' : 0,'female' : 1})

In [ ]:
df['Sex'].value_counts()

### One-Hot Encoding for Embarked

The Embarked column contains three categories representing the port where passengers boarded the Titanic.

We apply one-hot encoding to convert these categories into separate binary columns.

This prevents the model from assuming any ordinal relationship between the categories.

In [ ]:
df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)

In [ ]:
df.sample(3)

In [ ]:
dfc=df.corr(numeric_only=True)

plt.figure(figsize=(12,8))
sns.heatmap(dfc,annot=True)
plt.show()

### Relationship Between Sex and Survival

The Sex feature shows the strongest correlation with survival in the Titanic dataset.

Historically, during the Titanic disaster, the evacuation policy prioritized **women and children first**.

Because of this policy, female passengers had a significantly higher survival rate compared to male passengers.

This explains why the Sex feature becomes one of the most important predictors in the machine learning model.

In [ ]:
sns.countplot(x="Sex", hue="Survived", data=df)

plt.title("Survival Count by Gender")
plt.show()

In [ ]:
sns.countplot(x="Pclass", hue="Survived", data=df)

plt.title("Survival by Passenger Class")
plt.show()

In [ ]:
sns.histplot(df["Age"], bins=30, kde=True)

plt.title("Age Distribution")
plt.show()

In [ ]:
sns.boxplot(x="Survived", y="Age", data=df)
plt.show()

In [ ]:
X = df.drop("Survived", axis=1)
y = df["Survived"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_test

## Logistic Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_lr = accuracy_score(y_test, y_pred_lr)
accuracy_lr
print(classification_report(y_test, y_pred_lr))

## K-Nearest Neighbors (KNN) Model

KNN is a distance-based machine learning algorithm used for classification and regression.

Instead of learning explicit parameters like Logistic Regression, KNN works by:

1. Calculating the distance between a new data point and all training points
2. Selecting the **K nearest neighbors**
3. Assigning the majority class among those neighbors

Key characteristics:
- Non-parametric algorithm
- Works based on **distance metrics** (usually Euclidean distance)
- Sensitive to **feature scaling**, which is why we applied StandardScaler

For this model we start with:

K = 5

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

In [ ]:
knn_accuracy = accuracy_score(y_test, y_pred_knn)

print("KNN Accuracy:", knn_accuracy)
print(classification_report(y_test, y_pred_lr))

## Decision Tree Classifier

Decision Tree is a supervised machine learning algorithm used for both
classification and regression.

It works by splitting the dataset into smaller subsets based on feature values.

The model creates a tree structure consisting of:

- Root Node (starting point)
- Decision Nodes (feature-based splits)
- Leaf Nodes (final prediction)

Example intuition for Titanic dataset:

If Sex = Female → Higher survival probability  
If Sex = Male and Age > 30 → Lower survival probability

Decision Trees are powerful because they can capture **non-linear relationships**
between features and the target variable.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

In [ ]:
dt_accuracy = accuracy_score(y_test, y_pred_dt)

print("Decision Tree Accuracy:", dt_accuracy)
print(classification_report(y_test,y_pred_dt))

## Random Forest Classifier

Random Forest is an ensemble machine learning algorithm that combines
multiple Decision Trees to improve prediction performance.

Instead of relying on a single tree, Random Forest:

1. Creates many decision trees
2. Trains each tree on a random subset of the data
3. Combines predictions using majority voting

Advantages:
- Reduces overfitting compared to a single decision tree
- Handles non-linear relationships well
- Works very well on structured/tabular data

Random Forest is widely used in real-world machine learning problems.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

In [ ]:
rf_accuracy = accuracy_score(y_test, y_pred_rf)

print("Random Forest Accuracy:", rf_accuracy)
print(classification_report(y_test,y_pred_rf))

## Support Vector Machine (SVM)

Support Vector Machine (SVM) is a powerful supervised learning algorithm
used for classification and regression tasks.

The main idea of SVM is to find the **best boundary (hyperplane)** that
separates the classes.

For the Titanic dataset, SVM tries to find a boundary that best separates:

Survived = 1  
Not Survived = 0

Key concept:
SVM focuses on the **support vectors**, which are the data points closest
to the decision boundary.

Advantages:
- Works well for classification problems
- Effective in high-dimensional spaces
- Can create complex decision boundaries using kernels

In [60]:
from sklearn.svm import SVC

svm = SVC()
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

In [62]:
svm_accuracy = accuracy_score(y_test, y_pred_svm)

print("SVM Accuracy:", svm_accuracy)
print(classification_report(y_test,y_pred_svm))

SVM Accuracy: 0.8212290502793296
              precision    recall  f1-score   support

           0       0.81      0.90      0.86       105
           1       0.84      0.70      0.76        74

    accuracy                           0.82       179
   macro avg       0.83      0.80      0.81       179
weighted avg       0.82      0.82      0.82       179



In [63]:
model_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "KNN",
        "Decision Tree",
        "Random Forest",
        "SVM"
    ],
    "Accuracy": [
        accuracy_lr,
        knn_accuracy,
        dt_accuracy,
        rf_accuracy,
        svm_accuracy
    ]
})

model_results

,Model,Accuracy
0,Logistic Regression,0.810056
1,KNN,0.804469
2,Decision Tree,0.787709
3,Random Forest,0.798883
4,SVM,0.821229


## Confusion Matrix

A Confusion Matrix is used to evaluate the performance of a classification model.

It shows four important values:

True Positive (TP)  → Survived predicted correctly  
True Negative (TN)  → Not survived predicted correctly  
False Positive (FP) → Predicted survived but actually not  
False Negative (FN) → Predicted not survived but actually survived

This gives deeper insight into model performance beyond accuracy.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_svm)
print(cm)

[[95 10]
 [22 52]]


In [66]:
import pickle

with open("../Models/titanic_svm_model.pkl", "wb") as file:
    pickle.dump(svm, file)

print("Model saved successfully!")

Model saved successfully!
